# Micrograd: A Tiny Automatic Differentiation Engine

Based on Andrej Karpathy's educational implementation.
This notebook builds a complete neural network training pipeline from scratch
using only Python primitives and the math module.

Key goals:
- Understand how automatic differentiation (autodiff) works under the hood
- Build a scalar-valued computation graph that tracks gradients
- Construct neural network layers on top of this autodiff system
- Train a binary classifier on synthetic data using gradient descent
- Visualize EVERYTHING: graphs, boundaries, gradients, parameters

The entire implementation fits in ~150 lines of code, yet it contains all
the essential machinery of modern deep learning frameworks.

## Part 1: The Value Class

Core building block for automatic differentiation.
Every scalar value wraps a float and tracks:
- `data`: the raw numeric value
- `grad`: the gradient (partial derivative) of the final loss w.r.t. this value
- `_backward`: a function that accumulates gradients to its children via the chain rule
- `_prev`: the set of parent nodes in the computation graph
- `_op`: the operation that produced this value (e.g., '+', '*', 'tanh')

In [24]:
import math
import random
from typing import Set, Callable

class Value:
    """Scalar value with automatic differentiation support."""
    def __init__(self, data: float, _children: tuple = (), _op: str = ''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f'Value(data={self.data}, grad={self.grad})'

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only int/float powers"
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other): return self * other
    def __radd__(self, other): return self + other
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __truediv__(self, other): return self * (other ** -1)
    def __rtruediv__(self, other): return other * (self ** -1)
    def __neg__(self): return self * -1

    def tanh(self):
        x = self.data
        t = (math.exp(2 * x) - 1) / (math.exp(2 * x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t ** 2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'relu')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1.0
        for v in reversed(topo):
            v._backward()

    def zero_grad(self):
        self.grad = 0.0

In [25]:
# ---- Visualize the computation graph ----
try:
    import graphviz
    HAS_GRAPHVIZ = True
except ImportError:
    HAS_GRAPHVIZ = False

def draw_graph(root, filename='computation_graph', format='png'):
    """Render computation graph using graphviz.
    Shows data values, gradients, and operations."""
    if not HAS_GRAPHVIZ:
        print("Install graphviz: pip install graphviz")
        return
    
    nodes, edges = set(), set()
    
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    
    dot = graphviz.Digraph()
    
    for n in nodes:
        label = f"data={n.data:.4f}\
grad={n.grad:.4f}"
        if n._op:
            # Operation node
            dot.node(str(id(n)) + n._op, n._op, shape='circle', style='filled',
                     fillcolor='lightyellow', width='0.3', height='0.3')
            # Value node
            dot.node(str(id(n)), label, shape='box', style='filled,rounded',
                     fillcolor='lightblue' if n.grad != 0 else 'white')
            # Connect op -> value
            dot.edge(str(id(n)) + n._op, str(id(n)))
        else:
            # Leaf node (input)
            color = 'lightgreen' if n.grad != 0 else 'white'
            dot.node(str(id(n)), label, shape='box', style='filled,rounded',
                     fillcolor=color)
        
        # Connect children to operation
        if n._op:
            for child in n._prev:
                dot.edge(str(id(child)), str(id(n)) + n._op)
    
    dot.render(filename, format=format, cleanup=True)
    display(dot)

# Test: build expression a*b + a**2 and draw graph
a = Value(3.0)
b = Value(2.0)
c = a * b + a ** 2
c.backward()
print(f"a={a.data}, b={b.data}, c={c.data}")
print(f"dc/da={a.grad}, dc/db={b.grad}  (expected: 8, 3)")
draw_graph(c)

a=3.0, b=2.0, c=15.0
dc/da=8.0, dc/db=3.0  (expected: 8, 3)
Install graphviz: pip install graphviz


In [ ]:
# ---- Manual computation graph using matplotlib ----import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

def plot_computation_graph(root):
    """Draw computation graph with matplotlib.
    Shows data values, gradients, and operations."""
    levels = {}  # level -> list of nodes at that depth
    
    def assign_levels(v, lv=0):
        # Check if v already assigned to any level
        already = False
        for nodes in levels.values():
            if v in nodes:
                already = True
                break
        if not already:
            levels.setdefault(lv, []).append(v)
            for child in v._prev:
                assign_levels(child, lv + 1)
    assign_levels(root)
    
    max_level = max(levels.keys())
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.set_xlim(-0.5, max_level + 1.5)
    
    # Assign y positions (centered per level)
    y_pos = {}
    for lv, nodes in levels.items():
        n = len(nodes)
        for i, node in enumerate(nodes):
            y_pos[id(node)] = (n - 1) / 2 - i
    
    y_max = max(abs(v) for v in y_pos.values()) if y_pos else 1
    ax.set_ylim(-y_max - 1, y_max + 1)
    
    # Draw edges (child -> parent)
    for lv, nodes in levels.items():
        for node in nodes:
            for child in node._prev:
                child_lv = None
                for l, nl in levels.items():
                    if child in nl:
                        child_lv = l
                        break
                if child_lv is not None:
                    ax.annotate('', xy=(child_lv, y_pos[id(child)]),
                                xytext=(lv, y_pos[id(node)]),
                                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    
    # Draw nodes
    for lv, nodes in levels.items():
        for node in nodes:
            y = y_pos[id(node)]
            label = f"data={node.data:.2f}"
            if node._op:
                label += f"\nop={node._op}"
            if node.grad != 0:
                label += f"\ngrad={node.grad:.2f}"
            if not node._prev:
                color = 'lightgreen'
            elif node.grad != 0:
                color = 'lightyellow'
            else:
                color = 'white'
            ax.text(lv, y, label, ha='center', va='center',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=color, edgecolor='black'),
                    fontsize=8)
    
    ax.set_title('Value Computation Graph (data + gradients)')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

plot_computation_graph(c)


ModuleNotFoundError: No module named 'matplotlib'

## Part 2: Neural Network Layers

Building neural network components on top of the Value class:
- Neuron: a single neuron with weights, bias, and activation
- Layer: a collection of neurons (fully connected)
- MLP: a stack of layers forming a multi-layer perceptron

In [ ]:
class Neuron:
    """Single neuron: y = tanh(w1*x1 + w2*x2 + ... + wn*xn + b)."""
    def __init__(self, nin: int):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()

    def parameters(self):
        return self.w + [self.b]

class Layer:
    """Fully connected layer."""
    def __init__(self, nin: int, nout: int):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    """Multi-layer perceptron."""
    def __init__(self, nin: int, nouts: list):
        sizes = [nin] + nouts
        self.layers = [Layer(sizes[i], sizes[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]

# Create network: 2 inputs -> 16 hidden -> 1 output
net = MLP(2, [16, 1])
print(f'Network has {len(net.parameters())} parameters')

In [ ]:
# ---- Visualize the untrained network's weights ----
def plot_parameter_distribution(net, title='Parameter Distribution'):
    """Plot histogram of weights and biases per layer."""
    fig, axes = plt.subplots(1, len(net.layers), figsize=(5*len(net.layers), 4),
                             squeeze=False)
    for i, layer in enumerate(net.layers):
        ax = axes[0][i]
        params = layer.parameters()
        data = [p.data for p in params]
        ax.hist(data, bins=15, alpha=0.7, edgecolor='black')
        ax.axvline(0, color='red', linestyle='--', alpha=0.5)
        ax.set_title(f'Layer {i} ({len(params)} params)')
        ax.set_xlabel('Weight/Bias value')
        ax.set_ylabel('Count')
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_parameter_distribution(net, 'Before Training: Random Weights')

## Part 3: Data Generation & Visualization

Two Moons dataset - a classic non-linear classification problem.

In [ ]:
def generate_data(n=100, noise=0.1):
    """Generate two interleaving crescent moons."""
    X, y = [], []
    for i in range(n):
        t = i / n
        if i < n // 2:
            x = math.cos(math.pi * t) + random.gauss(0, noise)
            y_val = math.sin(math.pi * t) + random.gauss(0, noise)
            label = 1.0
        else:
            x = 1 - math.cos(math.pi * t) + random.gauss(0, noise)
            y_val = 0.5 - math.sin(math.pi * t) + random.gauss(0, noise)
            label = -1.0
        X.append([x, y_val])
        y.append(label)
    return X, y

X, y = generate_data(100)
print(f'Generated {len(X)} samples')

# ---- Scatter plot of the dataset ----
colors = ['orange' if yi > 0 else 'steelblue' for yi in y]
plt.figure(figsize=(8, 6))
plt.scatter([xi[0] for xi in X], [xi[1] for xi in X], c=colors,
            edgecolors='black', s=60, alpha=0.8)
plt.title('Two Moons Dataset', fontsize=15)
plt.xlabel('$x_1$', fontsize=12)
plt.ylabel('$x_2$', fontsize=12)
plt.grid(True, alpha=0.3)
plt.axis('equal')
orange_patch = mpatches.Patch(color='orange', label='Class +1 (upper moon)')
blue_patch = mpatches.Patch(color='steelblue', label='Class -1 (lower moon)')
plt.legend(handles=[orange_patch, blue_patch], fontsize=11)
plt.show()

## Part 4: Training with Full History Tracking

We track loss, parameters, and gradients at each epoch to visualize the learning process.

In [ ]:
# ---- Training with history tracking ----
def train(net, X, y, epochs=100, lr=0.01, track_every=1):
    """Train network and return full history for visualization."""
    history = {
        'loss': [],
        'params': [],  # snapshot of all param values at each epoch
        'grad_norms_per_layer': [],  # grad norm for each layer
        'acc': [],
        'preds_grid': [],  # predictions on a mesh grid (for decision boundary)
    }
    
    # Create mesh grid for decision boundary visualization
    x_min, x_max = -1.5, 2.5
    y_min, y_max = -1.0, 1.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 30),
                         np.linspace(y_min, y_max, 30))
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    
    for epoch in range(epochs):
        # Forward pass
        loss = None
        for xi, yi in zip(X, y):
            xi_vals = [Value(x) for x in xi]
            yi_val = Value(yi)
            pred = net(xi_vals)[0]
            diff = pred - yi_val
            sample_loss = diff * diff
            if loss is None:
                loss = sample_loss
            else:
                loss = loss + sample_loss
        loss = loss * (1.0 / len(X))
        history['loss'].append(loss.data)
        
        # Backward
        for p in net.parameters():
            p.grad = 0.0
        loss.backward()
        
        # Track grad norms per layer
        grad_norms = []
        for layer in net.layers:
            gs = [p.grad for p in layer.parameters()]
            grad_norms.append(sum(g**2 for g in gs) ** 0.5)
        history['grad_norms_per_layer'].append(grad_norms)
        
        # Update parameters
        for p in net.parameters():
            p.data -= lr * p.grad
        
        # Track parameter values
        if epoch % track_every == 0:
            history['params'].append([p.data for p in net.parameters()])
        
        # Track accuracy
        correct = 0
        for xi, yi in zip(X, y):
            xi_vals = [Value(x) for x in xi]
            pred = net(xi_vals)[0]
            if (pred.data > 0 and yi > 0) or (pred.data <= 0 and yi < 0):
                correct += 1
        history['acc'].append(correct / len(X))
        
        # Track decision boundary every 10 epochs
        if epoch % 10 == 0 or epoch == epochs - 1:
            grid_preds = []
            for gp in grid_points:
                gp_vals = [Value(gp[0]), Value(gp[1])]
                p = net(gp_vals)[0]
                grid_preds.append(p.data)
            history['preds_grid'].append(np.array(grid_preds).reshape(xx.shape))
            print(f'Epoch {epoch:3d}: loss={loss.data:.4f}, acc={history["acc"][-1]:.2f}')
        else:
            history['preds_grid'].append(None)
    
    print(f'Final: loss={history["loss"][-1]:.4f}, acc={history["acc"][-1]:.3f}')
    return history, (xx, yy)

# Fresh network for training
net = MLP(2, [16, 1])
history, (xx, yy) = train(net, X, y, epochs=100, lr=0.01)

In [ ]:
# ---- LOSS CURVE ----
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Loss over epochs
ax = axes[0]
ax.plot(history['loss'], color='crimson', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss (MSE)')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='gray', linestyle='--', alpha=0.3)

# Plot 2: Loss on log scale
ax = axes[1]
ax.semilogy(history['loss'], color='crimson', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Loss (Log Scale)')
ax.grid(True, alpha=0.3)

# Plot 3: Accuracy over epochs
ax = axes[2]
ax.plot(history['acc'], color='green', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Classification Accuracy')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.3, label='Perfect')

plt.tight_layout()
plt.show()

print(f"Initial loss: {history['loss'][0]:.4f}  |  Final loss: {history['loss'][-1]:.4f}")
print(f"Initial acc:  {history['acc'][0]:.2%}  |  Final acc:  {history['acc'][-1]:.2%}")

In [ ]:
# ---- DECISION BOUNDARY EVOLUTION ----
# Show how the decision boundary changes during training
n_frames = 6
selected_epochs = np.linspace(0, len(history['preds_grid']) - 1, n_frames, dtype=int)
selected_epochs = [e for e in selected_epochs if history['preds_grid'][e] is not None]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

colors = ['orange' if yi > 0 else 'steelblue' for yi in y]

for i, epoch_idx in enumerate(selected_epochs):
    if i >= len(axes):
        break
    ax = axes[i]
    pred_grid = history['preds_grid'][epoch_idx]
    if pred_grid is None:
        continue
    
    # Colored decision regions
    ax.contourf(xx, yy, pred_grid, levels=20, cmap='RdBu_r', alpha=0.6)
    # Decision boundary at 0
    ax.contour(xx, yy, pred_grid, levels=[0], colors='black', linewidths=2, linestyles='--')
    # Data points
    ax.scatter([xi[0] for xi in X], [xi[1] for xi in X], c=colors,
               edgecolors='black', s=40, alpha=0.9)
    ax.set_title(f'Epoch {epoch_idx}: loss={history["loss"][epoch_idx]:.3f}, acc={history["acc"][epoch_idx]:.2%}')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_xlim(-1.5, 2.5)
    ax.set_ylim(-1.0, 1.5)
    ax.set_aspect('equal')

fig.suptitle('Decision Boundary Evolution During Training', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# ---- FINAL DECISION BOUNDARY (full resolution) ----
# High-res decision boundary for the trained model
x_min, x_max = -1.5, 2.5
y_min, y_max = -1.0, 1.5
h = 0.05
xx_fine, yy_fine = np.meshgrid(np.arange(x_min, x_max, h),
                                np.arange(y_min, y_max, h))

grid_preds = []
for gp in np.c_[xx_fine.ravel(), yy_fine.ravel()]:
    gp_vals = [Value(gp[0]), Value(gp[1])]
    p = net(gp_vals)[0]
    grid_preds.append(p.data)
Z = np.array(grid_preds).reshape(xx_fine.shape)

fig, ax = plt.subplots(figsize=(9, 7))

# Color regions by prediction value (not just class)
cf = ax.contourf(xx_fine, yy_fine, Z, levels=30, cmap='RdBu_r', alpha=0.7, vmin=-1, vmax=1)
# Decision boundary at 0
cs = ax.contour(xx_fine, yy_fine, Z, levels=[0], colors='black', linewidths=3)
ax.clabel(cs, inline=True, fontsize=12, fmt='Decision Boundary')
# Confidence contours
cs2 = ax.contour(xx_fine, yy_fine, Z, levels=[-0.5, 0.5], colors='gray',
                 linewidths=1, linestyles=':', alpha=0.6)

# Data points
ax.scatter([xi[0] for xi in X], [xi[1] for xi in X], c=colors,
           edgecolors='black', s=80, alpha=0.9, zorder=5)

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect('equal')
ax.set_title('Trained Network Decision Boundary', fontsize=15)
ax.set_xlabel('$x_1$', fontsize=12)
ax.set_ylabel('$x_2$', fontsize=12)
plt.colorbar(cf, ax=ax, label='Network output value')
plt.tight_layout()
plt.show()

# Count points on each side
correct = sum(1 for xi, yi in zip(X, y) if (net([Value(xi[0]), Value(xi[1])])[0].data > 0 and yi > 0) or (net([Value(xi[0]), Value(xi[1])])[0].data <= 0 and yi < 0))
print(f"Test accuracy: {correct}/{len(X)} = {correct/len(X):.1%}")

In [ ]:
# ---- PARAMETER EVOLUTION DURING TRAINING ----
# Visualize how each parameter changes over time
params_history = np.array(history['params'])  # shape: (n_snapshots, n_params)
n_snapshots = params_history.shape[0]
n_params = params_history.shape[1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Plot 1: All parameter trajectories
ax = axes[0]
for i in range(min(n_params, 65)):
    ax.plot(params_history[:, i], alpha=0.3, linewidth=0.8)
ax.set_xlabel('Snapshot index')
ax.set_ylabel('Parameter value')
ax.set_title(f'All {n_params} parameter trajectories')
ax.grid(True, alpha=0.3)

# Plot 2: Mean abs value over time (for each layer)
ax = axes[1]
layer_sizes = []
idx = 0
for layer in net.layers:
    n = len(layer.parameters())
    layer_sizes.append((idx, idx + n))
    idx += n

colors_layer = ['crimson', 'royalblue']
for li, (start, end) in enumerate(layer_sizes):
    mean_abs = np.mean(np.abs(params_history[:, start:end]), axis=1)
    ax.plot(mean_abs, color=colors_layer[li], linewidth=2, label=f'Layer {li}')
ax.set_xlabel('Snapshot index')
ax.set_ylabel('Mean |param|')
ax.set_title('Parameter Magnitude per Layer')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 3: Parameter distribution at start vs end
ax = axes[2]
ax.hist(params_history[0, :], bins=30, alpha=0.5, color='gray', edgecolor='black',
        label=f'Epoch 0 (init)', density=True)
ax.hist(params_history[-1, :], bins=30, alpha=0.5, color='crimson', edgecolor='black',
        label=f'Epoch {len(history["loss"])-1} (trained)', density=True)
ax.set_xlabel('Parameter value')
ax.set_ylabel('Density')
ax.set_title('Parameter Distribution: Before vs After')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---- GRADIENT FLOW VISUALIZATION ----
# Show gradient norms per layer throughout training
grad_norms = np.array(history['grad_norms_per_layer'])  # (epochs, n_layers)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Gradient norm over training
ax = axes[0]
for li in range(grad_norms.shape[1]):
    ax.plot(grad_norms[:, li], color=colors_layer[li], linewidth=2,
            label=f'Layer {li}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Gradient L2 Norm')
ax.set_title('Gradient Norm per Layer (during training)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

# Plot 2: Gradient-to-parameter ratio (relative update size)
ax = axes[1]
lr = 0.01
for li, (start, end) in enumerate(layer_sizes):
    rel_update = grad_norms[:, li] / (np.mean(np.abs(params_history[:, start:end]), axis=1) + 1e-8)
    ax.plot(rel_update * lr, color=colors_layer[li], linewidth=2,
            label=f'Layer {li}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Relative update (lr * ||grad|| / ||param||)')
ax.set_title('Relative Parameter Update Size')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

print(f"Layer 0 final grad norm: {grad_norms[-1, 0]:.6f}")
print(f"Layer 1 final grad norm: {grad_norms[-1, 1]:.6f}")
ratio = grad_norms[-1, 0] / (grad_norms[-1, 1] + 1e-8)
print(f"Gradient ratio (L0/L1): {ratio:.3f} {'<-- healthy flow' if 0.1 < ratio < 10 else '<-- unbalanced flow!'}")

In [ ]:
# ---- WEIGHT VISUALIZATION AS HEATMAP ----
# Show the weight matrix of the first layer as a heatmap
# Each row = one neuron, each column = one input feature
layer0 = net.layers[0]
W0 = np.array([[n.w[0].data, n.w[1].data] for n in layer0.neurons])
b0 = np.array([n.b.data for n in layer0.neurons])

fig, axes = plt.subplots(1, 2, figsize=(10, 6))

# Weight heatmap
ax = axes[0]
im = ax.imshow(W0, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_xlabel('Input feature')
ax.set_ylabel('Neuron index')
ax.set_title('Hidden Layer Weights (16 x 2)')
ax.set_xticks([0, 1])
ax.set_xticklabels(['$x_1$ weight', '$x_2$ weight'])
plt.colorbar(im, ax=ax, label='Weight value')

# Biases as bar chart
ax = axes[1]
ax.bar(range(len(b0)), b0, color='steelblue', edgecolor='black')
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Neuron index')
ax.set_ylabel('Bias value')
ax.set_title('Hidden Layer Biases')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Output layer weights
layer1 = net.layers[0]
W1 = np.array([n.w[0].data for n in net.layers[1].neurons])  # 16 weights -> 1 output
b1 = net.layers[1].neurons[0].b.data

fig, axes = plt.subplots(1, 2, figsize=(10, 3))

ax = axes[0]
ax.bar(range(len(W1)), W1, color='royalblue', edgecolor='black')
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('Hidden neuron index')
ax.set_ylabel('Weight')
ax.set_title('Output Layer Weights (16 -> 1)')
ax.grid(True, alpha=0.3, axis='y')

ax = axes[1]
ax.bar(['Output bias'], [b1], color='royalblue', edgecolor='black', width=0.4)
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('Bias value')
ax.set_title('Output Layer Bias')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ---- NEURON ACTIVATION ANALYSIS ----
# See what each hidden neuron is doing - are they saturated?
def get_activations(net, X):
    """Record hidden layer activations for all data points."""
    activations = []
    for xi in X:
        xi_vals = [Value(xi[0]), Value(xi[1])]
        # Forward through hidden layer
        h = net.layers[0](xi_vals)
        activations.append([n.data for n in h])
    return np.array(activations)

hidden_acts = get_activations(net, X)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Plot 1: Heatmap of activations (neurons x samples)
ax = axes[0]
im = ax.imshow(hidden_acts.T, cmap='RdYlBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_xlabel('Sample index')
ax.set_ylabel('Neuron index')
ax.set_title('Hidden Neuron Activations')
plt.colorbar(im, ax=ax, label='tanh output')

# Plot 2: Distribution of activations
ax = axes[1]
ax.hist(hidden_acts.ravel(), bins=40, color='purple', alpha=0.7, edgecolor='black')
ax.axvline(-1, color='red', linestyle='--', alpha=0.5, label='Saturated -1')
ax.axvline(1, color='red', linestyle='--', alpha=0.5, label='Saturated +1')
ax.axvline(0, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Activation value')
ax.set_ylabel('Count')
ax.set_title('Activation Distribution')
ax.legend()

# Plot 3: Fraction of saturated neurons per sample
ax = axes[2]
frac_saturated = np.mean((np.abs(hidden_acts) > 0.9), axis=1)
ax.hist(frac_saturated, bins=20, color='orange', alpha=0.7, edgecolor='black')
ax.set_xlabel('Fraction of saturated neurons')
ax.set_ylabel('Number of samples')
ax.set_title('Saturation per Sample')
ax.axvline(0.5, color='red', linestyle='--', alpha=0.5,
           label='>50% saturated = dying gradients')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Mean activation: {hidden_acts.mean():.4f}")
print(f"Saturated neurons (|act| > 0.9): {np.mean(np.abs(hidden_acts) > 0.9):.1%}")
print(f"Dead neurons (always 0): {np.mean(np.all(np.abs(hidden_acts) < 1e-6, axis=1)):.1%}")

In [ ]:
# ---- CONTOUR OF EACH NEURON'S DECISION ----
# Show how individual hidden neurons partition the input space
h = 0.1
xx_small, yy_small = np.meshgrid(np.arange(-1.5, 2.5, h),
                                  np.arange(-1.0, 1.5, h))

n_neurons = len(net.layers[0].neurons)
cols = 4
rows = (n_neurons + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
axes = axes.ravel()

for ni in range(n_neurons):
    ax = axes[ni]
    neuron = net.layers[0].neurons[ni]
    
    # Compute this neuron's output over the grid
    Z_neuron = []
    for gp in np.c_[xx_small.ravel(), yy_small.ravel()]:
        act = sum((wi * vi for wi, vi in zip(neuron.w, [Value(gp[0]), Value(gp[1])])), neuron.b)
        Z_neuron.append(math.tanh(act.data))
    Z_neuron = np.array(Z_neuron).reshape(xx_small.shape)
    
    ax.contourf(xx_small, yy_small, Z_neuron, levels=20, cmap='RdBu_r', alpha=0.7, vmin=-1, vmax=1)
    ax.contour(xx_small, yy_small, Z_neuron, levels=[0], colors='black', linewidths=2)
    ax.scatter([xi[0] for xi in X], [xi[1] for xi in X], c=colors,
               edgecolors='black', s=15, alpha=0.6)
    ax.set_title(f'Neuron {ni}')
    ax.set_xlim(-1.5, 2.5)
    ax.set_ylim(-1.0, 1.5)
    ax.set_aspect('equal')
    ax.axis('off')

# Hide unused axes
for ni in range(n_neurons, len(axes)):
    axes[ni].axis('off')

fig.suptitle('Individual Hidden Neuron Decision Boundaries', fontsize=16)
plt.tight_layout()
plt.show()
print("Each hidden neuron learns a different linear decision boundary.")
print("The output layer combines these to form the non-linear boundary.")

In [ ]:
# ---- 3D VIEW OF THE OUTPUT SURFACE ----
from mpl_toolkits.mplot3d import Axes3D

h = 0.15
xx_3d, yy_3d = np.meshgrid(np.arange(-1.5, 2.5, h),
                            np.arange(-1.0, 1.5, h))

Z_3d = []
for gp in np.c_[xx_3d.ravel(), yy_3d.ravel()]:
    gp_vals = [Value(gp[0]), Value(gp[1])]
    Z_3d.append(net(gp_vals)[0].data)
Z_3d = np.array(Z_3d).reshape(xx_3d.shape)

fig = plt.figure(figsize=(12, 5))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
surf = ax1.plot_surface(xx_3d, yy_3d, Z_3d, cmap='RdBu_r',
                        alpha=0.8, linewidth=0, antialiased=True)
ax1.contour(xx_3d, yy_3d, Z_3d, levels=[0], colors='black', linewidths=3, zorder=10)
ax1.set_xlabel('$x_1$')
ax1.set_ylabel('$x_2$')
ax1.set_zlabel('Network output')
ax1.set_title('3D Output Surface', fontsize=14)
ax1.view_init(elev=25, azim=-60)

# 2D contour with data
ax2 = fig.add_subplot(122)
cf = ax2.contourf(xx_3d, yy_3d, Z_3d, levels=30, cmap='RdBu_r', alpha=0.7, vmin=-1, vmax=1)
cs = ax2.contour(xx_3d, yy_3d, Z_3d, levels=[0], colors='black', linewidths=3)
ax2.scatter([xi[0] for xi in X], [xi[1] for xi in X], c=colors,
           edgecolors='black', s=50, alpha=0.9, zorder=5)
ax2.set_xlabel('$x_1$')
ax2.set_ylabel('$x_2$')
ax2.set_title('Top-Down View', fontsize=14)
ax2.set_aspect('equal')
plt.colorbar(cf, ax=ax2, label='Output')

plt.tight_layout()
plt.show()

In [ ]:
# ---- COMPARE: LINEAR CLASSIFIER vs NEURAL NETWORK ----
# Train a simple linear classifier (no hidden layer) to show why depth matters
linear_net = MLP(2, [1])  # No hidden layer, just output neuron
print("Training linear classifier (2 inputs -> 1 output, no hidden layer)...")
linear_history, _ = train(linear_net, X, y, epochs=100, lr=0.01)

# Compute decision boundary for linear model
Z_lin = []
for gp in np.c_[xx_fine.ravel(), yy_fine.ravel()]:
    gp_vals = [Value(gp[0]), Value(gp[1])]
    Z_lin.append(linear_net(gp_vals)[0].data)
Z_lin = np.array(Z_lin).reshape(xx_fine.shape)

# Compare: linear vs neural network
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Linear
ax = axes[0]
ax.contourf(xx_fine, yy_fine, Z_lin, levels=30, cmap='RdBu_r', alpha=0.7, vmin=-1, vmax=1)
ax.contour(xx_fine, yy_fine, Z_lin, levels=[0], colors='black', linewidths=2)
ax.scatter([xi[0] for xi in X], [xi[1] for xi in X], c=colors,
           edgecolors='black', s=50, alpha=0.9)
ax.set_title(f'Linear Classifier (acc: {linear_history["acc"][-1]:.2%})', fontsize=13)
ax.set_xlim(-1.5, 2.5)
ax.set_ylim(-1.0, 1.5)
ax.set_aspect('equal')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')

# Neural Network
ax = axes[1]
ax.contourf(xx_fine, yy_fine, Z, levels=30, cmap='RdBu_r', alpha=0.7, vmin=-1, vmax=1)
ax.contour(xx_fine, yy_fine, Z, levels=[0], colors='black', linewidths=2)
ax.scatter([xi[0] for xi in X], [xi[1] for xi in X], c=colors,
           edgecolors='black', s=50, alpha=0.9)
ax.set_title(f'Neural Network 2-16-1 (acc: {history["acc"][-1]:.2%})', fontsize=13)
ax.set_xlim(-1.5, 2.5)
ax.set_ylim(-1.0, 1.5)
ax.set_aspect('equal')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')

plt.tight_layout()
plt.show()

print(f"Linear model final loss: {linear_history['loss'][-1]:.4f}, acc: {linear_history['acc'][-1]:.2%}")
print(f"Neural net final loss:   {history['loss'][-1]:.4f}, acc: {history['acc'][-1]:.2%}")
print()
print("The linear classifier draws a straight line - it CANNOT separate the moons.")
print("The neural network with a hidden layer learns a curved decision boundary.")

In [ ]:
# ---- LOSS LANDSCAPE VISUALIZATION ----
# Visualize the loss surface by varying two random parameters
print("Exploring the loss landscape...")

# Pick two random parameters from the network
all_params = net.parameters()
idx1, idx2 = random.sample(range(len(all_params)), 2)
p1_orig = all_params[idx1].data
p2_orig = all_params[idx2].data

# Scan a 2D grid around their current values
span = 1.0
n_steps = 25
p1_range = np.linspace(p1_orig - span, p1_orig + span, n_steps)
p2_range = np.linspace(p2_orig - span, p2_orig + span, n_steps)

# Compute loss at each point on the 2D grid
loss_surface = np.zeros((n_steps, n_steps))
for i, p1_val in enumerate(p1_range):
    for j, p2_val in enumerate(p2_range):
        # Temporarily set parameter values
        all_params[idx1].data = p1_val
        all_params[idx2].data = p2_val
        
        # Compute loss on a subset of data
        loss = None
        for xi, yi in zip(X[:20], y[:20]):  # use subset for speed
            xi_vals = [Value(xi[0]), Value(xi[1])]
            yi_val = Value(yi)
            pred = net(xi_vals)[0]
            diff = pred - yi_val
            sl = diff * diff
            loss = sl if loss is None else loss + sl
        loss_surface[i, j] = loss.data / 20 if loss else 0

# Restore
all_params[idx1].data = p1_orig
all_params[idx2].data = p2_orig

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
P1, P2 = np.meshgrid(p1_range, p2_range)

surf = ax.plot_surface(P1, P2, loss_surface.T, cmap='viridis',
                       alpha=0.9, linewidth=0, antialiased=True)
ax.scatter([p1_orig], [p2_orig], [loss_surface[n_steps//2, n_steps//2]],
           color='red', s=100, label='Current params', zorder=10)
ax.set_xlabel(f'Parameter {idx1} value')
ax.set_ylabel(f'Parameter {idx2} value')
ax.set_zlabel('Loss')
ax.set_title('2D Slice of the Loss Landscape', fontsize=14)
ax.view_init(elev=30, azim=-45)
plt.show()

print("The loss landscape is high-dimensional and non-convex.")
print("This 2D slice shows a small cross-section around the current parameters.")

## Part 5: Summary of What We've Learned

### Key Concepts

1. **Automatic Differentiation (Autodiff)**:
   - Every operation builds a node in a computation graph DAG
   - Forward pass computes values, backward pass computes gradients via chain rule
   - Time complexity: O(n) for both forward and backward
   - Space complexity: O(n) for the graph (can be optimized with checkpointing)

2. **Computation Graph**:
   - Nodes = Values, edges = data flow
   - Topological sort ensures correct order for backprop
   - Gradient accumulation (+=) handles nodes with multiple parents

3. **Neural Network Architecture**:
   - Neuron: weighted sum + bias + non-linear activation
   - Layer: parallel neurons
   - MLP: stacked layers for hierarchical feature learning
   - Hidden layer learns a distributed representation of the input

4. **Training Dynamics**:
   - Loss decreases (not always monotonically)
   - Decision boundary evolves from random to structured
   - Gradient norms decrease as we approach a minimum
   - Parameters move from initial distribution to task-specific values

5. **Why Non-linearity Matters**:
   - Linear classifiers can only draw straight decision boundaries
   - Hidden layers with tanh/ReLU enable curved boundaries
   - More neurons = more complex boundaries (but risk of overfitting)

### Visualizations Summary
| Visual | What it shows |
|--------|---------------|
| Computation Graph | The DAG of operations with data and gradients |
| Loss Curve | Training progress over epochs |
| Decision Boundary | How the network partitions the input space |
| Parameter Trajectories | How weights evolve during training |
| Gradient Norms | Gradient flow health (vanishing/exploding) |
| Neuron Activations | Are neurons saturated or dead? |
| Loss Landscape | 2D slice of the high-dimensional loss surface |
| Linear vs NN | Why depth and non-linearity are essential |